# Ablation_model: all 4 full-participation (p=1.0) scalability checks, merged

**This notebook merges the 4 standalone ablation notebooks
(`ABLATION_TINY_CNN_MNIST`, `ABLATION_TINY_CNN_FEMNIST`,
`ABLATION_SMALL_RESNET_MNIST`, `ABLATION_SMALL_RESNET_FEMNIST`) into
one file**, per explicit request (2026-11-18) -- same purpose as each
of those (see `ARCHITECTURE.md` SS5.5.1): proving the protocol is
*capable* of training with all 50 clients real (zero dummies), not
merely configured to avoid it, for the reviewer question "does it only
work at `sampling_rate=0.25`?" 2 rounds per combo, not 20 -- this is a
correctness/scalability check, not a convergence experiment.

**What's deduped vs. kept separate, and why**: clone/install/sanity/
config/ring setup are IDENTICAL across all 4 combos (same repo, same
`CFG`, same 50-client crypto ring) -- run ONCE, not 4 times. Each
dataset (MNIST, FEMNIST) is loaded ONCE and reused by both models that
use it. What differs per combo -- the model itself, and the checkpoint
branch -- is rebuilt fresh for each of the 4 sections below. Every cell
that *can* be shared is imported verbatim from the same
`notebooks/_build_mvp_run_notebooks.py` / `notebooks/ablation_study/
_build_ablation_notebooks.py` the 4 standalone notebooks use -- nothing
here is a reimplementation, except the checkpoint-setup cell, which
needed one real change to work correctly across 4 branch-switches in a
single session (see that cell's own comment for exactly why).

**Run order**: tiny_cnn+MNIST -> tiny_cnn+FEMNIST -> small_resnet+MNIST
-> small_resnet+FEMNIST. Each combo pushes to its own dedicated
checkpoint branch (`colab-checkpoints-ablation-case{{1,2}}-{{model}}`),
exactly as the standalone notebooks did -- merging the notebook file
does NOT merge the checkpoint branches.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


## Load both datasets once (reused across all 4 combos below)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 5. Load FEMNIST, writer-disjoint non-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.femnist.cache_dir points at a
# Drive path this run's Drive doesn't have mounted -- overridden here
# to a local Colab cache dir instead, same no-Drive pattern the MNIST
# run uses for its "source" override. flwr-datasets downloads and
# caches FEMNIST (~200MB) to this path over the network, no Drive
# dependency at all.
from fl.data_noniid import prepare_case2

case2_data = prepare_case2(seed=CFG['seed'], cache_dir="/content/femnist_cache/")
print("clients:", len(case2_data['client_train']))
print("server_root writers:", len(case2_data['split'].server_root_writers))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))


## Combo 1/4: tiny_cnn + MNIST  (branch: `colab-checkpoints-ablation-case1-tiny_cnn`)

In [ ]:
# 7. Build the model.
# ~20.6K params, matches WhiteCloak/PQS-BFL's model scale (see fl/models.py build_tiny_cnn docstring).
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# Switch this session's checkpoint clone to THIS combo's own branch.
#
# Unlike the single-combo ablation notebooks' checkpoint-setup cell,
# this version runs the fetch-or-create-branch step EVERY time it's
# called (once per combo in this merged notebook), not only on first
# clone -- otherwise combo 2, 3, and 4 would silently keep pushing to
# combo 1's branch (see this script's own module docstring).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
CKPT_BRANCH = "colab-checkpoints-ablation-case1-tiny_cnn"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    del _token
else:
    print("checkpoint clone already present this session (from an earlier combo) -- reusing it, switching branch below.")

# Runs every call (not just on first clone) -- see module docstring.
_fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
if _fetch.returncode == 0:
    _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
    print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
else:
    # Explicit start point (origin/main), NOT "current HEAD" -- on combo
    # 2+, HEAD is whatever the previous combo's branch left it at, and
    # branching from there would carry that combo's checkpoint history
    # into a branch that should start clean.
    _git(["fetch", "origin", "main:refs/remotes/origin/main"], check=False)
    _git(["checkout", "-B", CKPT_BRANCH, "origin/main"])
    print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh from main.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """Same contract as the per-notebook version (fl/round_loop.py's
    on_checkpoint_saved) -- never raises; a push failure costs a retry
    next round, not the training run."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The ablation run: FORCE full participation (p=1.0), 2 rounds only.
#
# This is the ONLY cell that differs from the main
# RUN_MVP_TINY_CNN_MNIST.ipynb notebook. Everything
# above and below this cell (clone/install/sanity/config/dataset/ring/
# model/checkpoint-setup/eval) is the exact same code, imported
# verbatim from notebooks/_build_mvp_run_notebooks.py -- see this
# notebook's own intro markdown and ARCHITECTURE.md SS5.5.1.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

# FIX.md F21 (2026-11-17): the ablation's whole point -- override the
# shipped config's sampling_rate (0.25) to 1.0 so the VRF self-sampling
# gate (fl/client.py's vrf_is_sampled, ARCHITECTURE.md SS5.5) passes for
# EVERY one of the 50 clients. Zero dummy (q_k=0) submissions this run;
# every accepted submission is a real, locally-trained delta. This is a
# config mutation in the notebook's own CFG dict, not a library change --
# run_training_loop reads CFG['protocol']['sampling_rate'] fresh every
# call (fl/round_loop.py), so nothing in fl/ needs to know this is an
# ablation.
CFG['protocol']['sampling_rate'] = 1.0
ABLATION_SAMPLING_RATE = CFG['protocol']['sampling_rate']
REAL_ROUNDS = 2  # deliberately NOT 20 -- see intro markdown for why.

# '_ablation_p1_r2' suffix: distinct from every main-matrix and
# attack-matrix checkpoint filename by construction, so this can never
# be mistaken for, or accidentally resumed as, a real training run.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_ablation_p1_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"ablation-bootstrap-{REAL_MODEL}-case1".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found ablation checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()

print("="*70)
print(f"ABLATION (p={ABLATION_SAMPLING_RATE}) RESULT -- {REAL_MODEL} + MNIST")
print("="*70)
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')} <-- UNEXPECTED at p=1.0, report this")
    else:
        n_acc = h.get('accepted')
        flag = "OK (all 50 real, no dummies)" if n_acc == n_clients else f"UNEXPECTED: only {n_acc}/{n_clients} accepted at p=1.0"
        print(f"  round {h.get('round')}: accepted={n_acc}/{n_clients}, aggregate_exact={h.get('aggregate_exact')} -- {flag}")

n_ok_rounds = sum(1 for h in real_history if not h.get('aborted') and h.get('accepted') == n_clients)
if n_ok_rounds == REAL_ROUNDS:
    print(f"\nPASS: all {REAL_ROUNDS} rounds completed with all {n_clients} clients accepted as real contributors -- "
          f"the protocol is NOT structurally limited to p<1.0, subsampling is a config choice (ARCHITECTURE.md SS5.5.1).")
else:
    print(f"\nINCONCLUSIVE/FAIL: only {n_ok_rounds}/{REAL_ROUNDS} rounds had all {n_clients} clients accepted -- investigate before citing this as the full-participation proof.")


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")


## Combo 2/4: tiny_cnn + FEMNIST  (branch: `colab-checkpoints-ablation-case2-tiny_cnn`)

In [ ]:
# 7. Build the model.
# ~20.6K params, matches WhiteCloak/PQS-BFL's model scale (see fl/models.py build_tiny_cnn docstring) -- same backbone as the MNIST ablation, different (non-IID, writer-split) dataset.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# Switch this session's checkpoint clone to THIS combo's own branch.
#
# Unlike the single-combo ablation notebooks' checkpoint-setup cell,
# this version runs the fetch-or-create-branch step EVERY time it's
# called (once per combo in this merged notebook), not only on first
# clone -- otherwise combo 2, 3, and 4 would silently keep pushing to
# combo 1's branch (see this script's own module docstring).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
CKPT_BRANCH = "colab-checkpoints-ablation-case2-tiny_cnn"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    del _token
else:
    print("checkpoint clone already present this session (from an earlier combo) -- reusing it, switching branch below.")

# Runs every call (not just on first clone) -- see module docstring.
_fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
if _fetch.returncode == 0:
    _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
    print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
else:
    # Explicit start point (origin/main), NOT "current HEAD" -- on combo
    # 2+, HEAD is whatever the previous combo's branch left it at, and
    # branching from there would carry that combo's checkpoint history
    # into a branch that should start clean.
    _git(["fetch", "origin", "main:refs/remotes/origin/main"], check=False)
    _git(["checkout", "-B", CKPT_BRANCH, "origin/main"])
    print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh from main.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """Same contract as the per-notebook version (fl/round_loop.py's
    on_checkpoint_saved) -- never raises; a push failure costs a retry
    next round, not the training run."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The ablation run: FORCE full participation (p=1.0), 2 rounds only.
#
# This is the ONLY cell that differs from the main
# RUN_MVP_TINY_CNN_FEMNIST.ipynb notebook. Everything
# above and below this cell (clone/install/sanity/config/dataset/ring/
# model/checkpoint-setup/eval) is the exact same code, imported
# verbatim from notebooks/_build_mvp_run_notebooks.py -- see this
# notebook's own intro markdown and ARCHITECTURE.md SS5.5.1.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

# FIX.md F21 (2026-11-17): the ablation's whole point -- override the
# shipped config's sampling_rate (0.25) to 1.0 so the VRF self-sampling
# gate (fl/client.py's vrf_is_sampled, ARCHITECTURE.md SS5.5) passes for
# EVERY one of the 50 clients. Zero dummy (q_k=0) submissions this run;
# every accepted submission is a real, locally-trained delta. This is a
# config mutation in the notebook's own CFG dict, not a library change --
# run_training_loop reads CFG['protocol']['sampling_rate'] fresh every
# call (fl/round_loop.py), so nothing in fl/ needs to know this is an
# ablation.
CFG['protocol']['sampling_rate'] = 1.0
ABLATION_SAMPLING_RATE = CFG['protocol']['sampling_rate']
REAL_ROUNDS = 2  # deliberately NOT 20 -- see intro markdown for why.

# '_ablation_p1_r2' suffix: distinct from every main-matrix and
# attack-matrix checkpoint filename by construction, so this can never
# be mistaken for, or accidentally resumed as, a real training run.
CKPT_PATH = f'{CKPT_DIR}/case2_{REAL_MODEL}_ablation_p1_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"ablation-bootstrap-{REAL_MODEL}-case2".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found ablation checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()

print("="*70)
print(f"ABLATION (p={ABLATION_SAMPLING_RATE}) RESULT -- {REAL_MODEL} + FEMNIST")
print("="*70)
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')} <-- UNEXPECTED at p=1.0, report this")
    else:
        n_acc = h.get('accepted')
        flag = "OK (all 50 real, no dummies)" if n_acc == n_clients else f"UNEXPECTED: only {n_acc}/{n_clients} accepted at p=1.0"
        print(f"  round {h.get('round')}: accepted={n_acc}/{n_clients}, aggregate_exact={h.get('aggregate_exact')} -- {flag}")

n_ok_rounds = sum(1 for h in real_history if not h.get('aborted') and h.get('accepted') == n_clients)
if n_ok_rounds == REAL_ROUNDS:
    print(f"\nPASS: all {REAL_ROUNDS} rounds completed with all {n_clients} clients accepted as real contributors -- "
          f"the protocol is NOT structurally limited to p<1.0, subsampling is a config choice (ARCHITECTURE.md SS5.5.1).")
else:
    print(f"\nINCONCLUSIVE/FAIL: only {n_ok_rounds}/{REAL_ROUNDS} rounds had all {n_clients} clients accepted -- investigate before citing this as the full-participation proof.")


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# unseen_test: writers held out entirely from every client/server_root/
# global_val role (ROADMAP.md §3.4) -- THE headline number for FEMNIST,
# since it's the only split that tests generalization to writers the
# federation never saw during training at all, non-IID handwriting
# style included.
unseen_ds = case2_data['unseen_test']
unseen_acc = evaluate_accuracy(model, unseen_ds.images, unseen_ds.labels)

# global_val: held-out SEEN-writer samples (disjoint per-client holdout,
# not disjoint writers) -- secondary sanity check, not the headline
# number, same role as MNIST's global_val.
val_ds = case2_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: unseen-writer FEMNIST test accuracy = {unseen_acc:.4f} ({len(unseen_ds.images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")


## Combo 3/4: small_resnet + MNIST  (branch: `colab-checkpoints-ablation-case1-small_resnet`)

In [ ]:
# 7. Build the model.
# ~263K params, anchored to WhiteCloak's own second reported model (CIFAR-10-L ResNet, 273K params -- see fl/models.py build_small_resnet docstring).
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'small_resnet'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# Switch this session's checkpoint clone to THIS combo's own branch.
#
# Unlike the single-combo ablation notebooks' checkpoint-setup cell,
# this version runs the fetch-or-create-branch step EVERY time it's
# called (once per combo in this merged notebook), not only on first
# clone -- otherwise combo 2, 3, and 4 would silently keep pushing to
# combo 1's branch (see this script's own module docstring).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
CKPT_BRANCH = "colab-checkpoints-ablation-case1-small_resnet"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    del _token
else:
    print("checkpoint clone already present this session (from an earlier combo) -- reusing it, switching branch below.")

# Runs every call (not just on first clone) -- see module docstring.
_fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
if _fetch.returncode == 0:
    _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
    print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
else:
    # Explicit start point (origin/main), NOT "current HEAD" -- on combo
    # 2+, HEAD is whatever the previous combo's branch left it at, and
    # branching from there would carry that combo's checkpoint history
    # into a branch that should start clean.
    _git(["fetch", "origin", "main:refs/remotes/origin/main"], check=False)
    _git(["checkout", "-B", CKPT_BRANCH, "origin/main"])
    print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh from main.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """Same contract as the per-notebook version (fl/round_loop.py's
    on_checkpoint_saved) -- never raises; a push failure costs a retry
    next round, not the training run."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The ablation run: FORCE full participation (p=1.0), 2 rounds only.
#
# This is the ONLY cell that differs from the main
# RUN_MVP_SMALL_RESNET_MNIST.ipynb notebook. Everything
# above and below this cell (clone/install/sanity/config/dataset/ring/
# model/checkpoint-setup/eval) is the exact same code, imported
# verbatim from notebooks/_build_mvp_run_notebooks.py -- see this
# notebook's own intro markdown and ARCHITECTURE.md SS5.5.1.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

# FIX.md F21 (2026-11-17): the ablation's whole point -- override the
# shipped config's sampling_rate (0.25) to 1.0 so the VRF self-sampling
# gate (fl/client.py's vrf_is_sampled, ARCHITECTURE.md SS5.5) passes for
# EVERY one of the 50 clients. Zero dummy (q_k=0) submissions this run;
# every accepted submission is a real, locally-trained delta. This is a
# config mutation in the notebook's own CFG dict, not a library change --
# run_training_loop reads CFG['protocol']['sampling_rate'] fresh every
# call (fl/round_loop.py), so nothing in fl/ needs to know this is an
# ablation.
CFG['protocol']['sampling_rate'] = 1.0
ABLATION_SAMPLING_RATE = CFG['protocol']['sampling_rate']
REAL_ROUNDS = 2  # deliberately NOT 20 -- see intro markdown for why.

# '_ablation_p1_r2' suffix: distinct from every main-matrix and
# attack-matrix checkpoint filename by construction, so this can never
# be mistaken for, or accidentally resumed as, a real training run.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_ablation_p1_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"ablation-bootstrap-{REAL_MODEL}-case1".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found ablation checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()

print("="*70)
print(f"ABLATION (p={ABLATION_SAMPLING_RATE}) RESULT -- {REAL_MODEL} + MNIST")
print("="*70)
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')} <-- UNEXPECTED at p=1.0, report this")
    else:
        n_acc = h.get('accepted')
        flag = "OK (all 50 real, no dummies)" if n_acc == n_clients else f"UNEXPECTED: only {n_acc}/{n_clients} accepted at p=1.0"
        print(f"  round {h.get('round')}: accepted={n_acc}/{n_clients}, aggregate_exact={h.get('aggregate_exact')} -- {flag}")

n_ok_rounds = sum(1 for h in real_history if not h.get('aborted') and h.get('accepted') == n_clients)
if n_ok_rounds == REAL_ROUNDS:
    print(f"\nPASS: all {REAL_ROUNDS} rounds completed with all {n_clients} clients accepted as real contributors -- "
          f"the protocol is NOT structurally limited to p<1.0, subsampling is a config choice (ARCHITECTURE.md SS5.5.1).")
else:
    print(f"\nINCONCLUSIVE/FAIL: only {n_ok_rounds}/{REAL_ROUNDS} rounds had all {n_clients} clients accepted -- investigate before citing this as the full-participation proof.")


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")


## Combo 4/4: small_resnet + FEMNIST  (branch: `colab-checkpoints-ablation-case2-small_resnet`)

In [ ]:
# 7. Build the model.
# ~263K params (see fl/models.py build_small_resnet docstring) on the non-IID, writer-split FEMNIST dataset -- completes the 2-model x 2-dataset ablation matrix.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'small_resnet'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# Switch this session's checkpoint clone to THIS combo's own branch.
#
# Unlike the single-combo ablation notebooks' checkpoint-setup cell,
# this version runs the fetch-or-create-branch step EVERY time it's
# called (once per combo in this merged notebook), not only on first
# clone -- otherwise combo 2, 3, and 4 would silently keep pushing to
# combo 1's branch (see this script's own module docstring).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
CKPT_BRANCH = "colab-checkpoints-ablation-case2-small_resnet"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    del _token
else:
    print("checkpoint clone already present this session (from an earlier combo) -- reusing it, switching branch below.")

# Runs every call (not just on first clone) -- see module docstring.
_fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
if _fetch.returncode == 0:
    _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
    print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
else:
    # Explicit start point (origin/main), NOT "current HEAD" -- on combo
    # 2+, HEAD is whatever the previous combo's branch left it at, and
    # branching from there would carry that combo's checkpoint history
    # into a branch that should start clean.
    _git(["fetch", "origin", "main:refs/remotes/origin/main"], check=False)
    _git(["checkout", "-B", CKPT_BRANCH, "origin/main"])
    print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh from main.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """Same contract as the per-notebook version (fl/round_loop.py's
    on_checkpoint_saved) -- never raises; a push failure costs a retry
    next round, not the training run."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The ablation run: FORCE full participation (p=1.0), 2 rounds only.
#
# This is the ONLY cell that differs from the main
# RUN_MVP_SMALL_RESNET_FEMNIST.ipynb notebook. Everything
# above and below this cell (clone/install/sanity/config/dataset/ring/
# model/checkpoint-setup/eval) is the exact same code, imported
# verbatim from notebooks/_build_mvp_run_notebooks.py -- see this
# notebook's own intro markdown and ARCHITECTURE.md SS5.5.1.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

# FIX.md F21 (2026-11-17): the ablation's whole point -- override the
# shipped config's sampling_rate (0.25) to 1.0 so the VRF self-sampling
# gate (fl/client.py's vrf_is_sampled, ARCHITECTURE.md SS5.5) passes for
# EVERY one of the 50 clients. Zero dummy (q_k=0) submissions this run;
# every accepted submission is a real, locally-trained delta. This is a
# config mutation in the notebook's own CFG dict, not a library change --
# run_training_loop reads CFG['protocol']['sampling_rate'] fresh every
# call (fl/round_loop.py), so nothing in fl/ needs to know this is an
# ablation.
CFG['protocol']['sampling_rate'] = 1.0
ABLATION_SAMPLING_RATE = CFG['protocol']['sampling_rate']
REAL_ROUNDS = 2  # deliberately NOT 20 -- see intro markdown for why.

# '_ablation_p1_r2' suffix: distinct from every main-matrix and
# attack-matrix checkpoint filename by construction, so this can never
# be mistaken for, or accidentally resumed as, a real training run.
CKPT_PATH = f'{CKPT_DIR}/case2_{REAL_MODEL}_ablation_p1_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"ablation-bootstrap-{REAL_MODEL}-case2".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found ablation checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()

print("="*70)
print(f"ABLATION (p={ABLATION_SAMPLING_RATE}) RESULT -- {REAL_MODEL} + FEMNIST")
print("="*70)
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')} <-- UNEXPECTED at p=1.0, report this")
    else:
        n_acc = h.get('accepted')
        flag = "OK (all 50 real, no dummies)" if n_acc == n_clients else f"UNEXPECTED: only {n_acc}/{n_clients} accepted at p=1.0"
        print(f"  round {h.get('round')}: accepted={n_acc}/{n_clients}, aggregate_exact={h.get('aggregate_exact')} -- {flag}")

n_ok_rounds = sum(1 for h in real_history if not h.get('aborted') and h.get('accepted') == n_clients)
if n_ok_rounds == REAL_ROUNDS:
    print(f"\nPASS: all {REAL_ROUNDS} rounds completed with all {n_clients} clients accepted as real contributors -- "
          f"the protocol is NOT structurally limited to p<1.0, subsampling is a config choice (ARCHITECTURE.md SS5.5.1).")
else:
    print(f"\nINCONCLUSIVE/FAIL: only {n_ok_rounds}/{REAL_ROUNDS} rounds had all {n_clients} clients accepted -- investigate before citing this as the full-participation proof.")


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# unseen_test: writers held out entirely from every client/server_root/
# global_val role (ROADMAP.md §3.4) -- THE headline number for FEMNIST,
# since it's the only split that tests generalization to writers the
# federation never saw during training at all, non-IID handwriting
# style included.
unseen_ds = case2_data['unseen_test']
unseen_acc = evaluate_accuracy(model, unseen_ds.images, unseen_ds.labels)

# global_val: held-out SEEN-writer samples (disjoint per-client holdout,
# not disjoint writers) -- secondary sanity check, not the headline
# number, same role as MNIST's global_val.
val_ds = case2_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: unseen-writer FEMNIST test accuracy = {unseen_acc:.4f} ({len(unseen_ds.images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")


## Summary

All 4 combos above ran 2 rounds each at `sampling_rate=1.0` (every one
of the 50 clients real, zero dummy submissions). Each combo printed its
own PASS/FAIL line; scroll up to confirm all 4 said PASS before citing
this as the full-participation proof in the paper (ARCHITECTURE.md
SS5.5.1). Each combo also wrote its own checkpoint to its own dedicated
branch -- resuming any one of them independently is still possible,
exactly as if the 4 standalone notebooks had been run separately.
